# Structural file conversion

This tutorial shows small local conversions among NDB, CNDB, PDB, and simple text SpaceWalk files using CNDBTools. The converters are meant for local interoperability workflows and small examples. They do not download remote files.


## Imports and workspace

The example writes all temporary files into a local temporary directory.

In [ ]:
from pathlib import Path
import tempfile

import h5py
import numpy as np

from OpenMiChroM.CndbTools import CndbTools, convert_structure_file
from OpenMiChroM._structural_io.readers import NDBTextReader

workdir = Path(tempfile.mkdtemp(prefix="openmichrom-conversion-"))
workdir

## Create a tiny NDB file

A real NDB file may contain many beads and frames. This tiny file has two frames and three beads so that it is easy to inspect.

In [ ]:
ndb_path = workdir / "tiny.ndb"
ndb_path.write_text(
    "HEADER    NDB File generated by OpenMiChroM\n"
    "TITLE     tiny example\n"
    "SEQCHR   1 C1     3  A1 B1 A2\n"
    "MODEL 1\n"
    "CHROM         1 A1 C1         1      0.000      1.000      2.000          1      50000    0.000\n"
    "CHROM         2 B1 C1         2      3.000      4.000      5.000      50001     100000    0.000\n"
    "CHROM         3 A2 C1         3      6.000      7.000      8.000     100001     150000    0.000\n"
    "ENDMDL\n"
    "MODEL 2\n"
    "CHROM         1 A1 C1         1     10.000     11.000     12.000          1      50000    0.000\n"
    "CHROM         2 B1 C1         2     13.000     14.000     15.000      50001     100000    0.000\n"
    "CHROM         3 A2 C1         3     16.000     17.000     18.000     100001     150000    0.000\n"
    "ENDMDL\n"
    "END\n",
    encoding="utf-8",
)

reader = NDBTextReader(ndb_path)
print("frames:", reader.frame_ids)
print("types:", reader.types)
print("frame 1 shape:", reader.get_coordinates(1).shape)

## Convert NDB to CNDB

CNDB output uses the new stream-friendly CNDB v2 writer by default, including `/Header`, `/_index`, and `_index_offset`.

In [ ]:
cndb_path = workdir / "tiny.cndb"
convert_structure_file(ndb_path, cndb_path)

with h5py.File(cndb_path, "r") as h5:
    print("root keys:", list(h5.keys()))
    print("format:", h5["Header"].attrs["format_name"], h5["Header"].attrs["format_version"])
    print("has _index_offset:", "_index_offset" in h5.attrs)
    print("frame 1 shape:", h5["1"].shape)

## Convert only selected frames and beads

For local files, you can select frames and a bead window before writing the output. This keeps small interoperability exports from accidentally becoming large rewrites.

In [ ]:
subset_ndb = workdir / "tiny_subset.ndb"
convert_structure_file(
    cndb_path,
    subset_ndb,
    frames=[2],
    start=1,
    stop=3,
)

subset = NDBTextReader(subset_ndb)
print("frames:", subset.frame_ids)
print("types:", subset.types)
print(subset.get_coordinates(2))

## Convert CNDB back to NDB

This is useful when a downstream tool expects text NDB records.

In [ ]:
roundtrip_ndb = workdir / "tiny_roundtrip.ndb"
CndbTools.convert(cndb_path, roundtrip_ndb)

roundtrip = NDBTextReader(roundtrip_ndb)
print("frames:", roundtrip.frame_ids)
print("types:", roundtrip.types)
np.testing.assert_allclose(roundtrip.get_coordinates(2), reader.get_coordinates(2))

## Convert NDB to PDB

The PDB converter writes simple CA-like bead records. Type labels are mapped approximately to residue names for visualization-oriented workflows.

In [ ]:
pdb_path = workdir / "tiny.pdb"
convert_structure_file(ndb_path, pdb_path)

preview = "\n".join(pdb_path.read_text(encoding="utf-8").splitlines()[:6])
print(preview)

You can also customize the simple PDB bead fields for visualization workflows.

In [ ]:
custom_pdb = workdir / "tiny_custom.pdb"
convert_structure_file(
    ndb_path,
    custom_pdb,
    pdb_atom_name="BB",
    pdb_residue_name="CHR",
    pdb_chain_id="B",
)
print("\n".join(custom_pdb.read_text(encoding="utf-8").splitlines()[:4]))

## Convert simple text SpaceWalk files

OpenMiChroM supports a small clean-room SpaceWalk text dialect with `trace` sections and `chromosome start end x y z` coordinate rows. Text SpaceWalk files do not carry OpenMiChroM type labels, so converted beads are assigned `UN`.

In [ ]:
spw_path = workdir / "tiny.spw"
spw_path.write_text(
    "##format=sw1 name=tiny_spacewalk genome=hg38\n"
    "chromosome\tstart\tend\tx\ty\tz\n"
    "trace 0\n"
    "chr21\t1\t50000\t0.0\t1.0\t2.0\n"
    "chr21\t50001\t100000\t3.0\t4.0\t5.0\n"
    "trace 1\n"
    "chr21\t1\t50000\t10.0\t11.0\t12.0\n"
    "chr21\t50001\t100000\t13.0\t14.0\t15.0\n",
    encoding="utf-8",
)

spw_ndb = workdir / "from_spacewalk.ndb"
convert_structure_file(spw_path, spw_ndb)
spw_reader = NDBTextReader(spw_ndb)
print("frames:", spw_reader.frame_ids)
print("types:", spw_reader.types)
print(spw_reader.get_coordinates(1))

## Memory guard

The converter refuses selected payloads larger than `max_memory_mb` unless you explicitly pass `allow_large=True`. This tutorial uses a tiny threshold to show the error path.

In [ ]:
try:
    convert_structure_file(ndb_path, workdir / "too_small_limit.cndb", max_memory_mb=0)
except ValueError as exc:
    print(str(exc).split(". ")[0])

## Notes

- Converters are local-file only.
- CNDB output is indexed by default.
- Frame and bead filters can keep conversions small.
- PDB conversion is intentionally simple and approximate.
- Text SpaceWalk conversion supports simple `trace` files with `chromosome start end x y z` rows.
- Large production conversions should be explicit; the converter has a memory guard but still uses an in-memory trajectory representation.
